<a href="https://colab.research.google.com/github/unmtransinfo/drugcentral-tools/blob/master/python/colab/DrugCentral_Inxight_Bioactivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DrugCentral - Inxight Integration - BIOACTIVITY

Merging DC and Inxight bioactivity data via compound UNII IDs and protein UNIPROT IDs.

Note, from Inxight website: "The full database contains approximately 4,500 drugs, including FDA-approved, previously approved, over-the-counter, and investigational small-molecule and peptide drugs." Thus we expect many Inxight drugs will not be present in DrugCentral, which is scoped specifically with
*only* approved drugs.

* Inxight data downloaded from: https://drugs.ncats.io/downloads-public
* Inxight bioactivity file from: Jessica Maine, NCATS Informatics Core
  * Generated by Python script export_inxight_activity_targets_standalone.py
* DrugCentral data available from: https://drugcentral.org/download


Note that DrugCentral targets may be multi-component, single proteins as components. Thus Inxight targets are mapped to DC components.

In [1]:
import sys,os,re
import numpy as np
!pip install --upgrade pandas>=3.0.0
import pandas as pd
from google.colab import drive as colab_drive, auth as colab_auth
import gspread
from google.auth import default as g_auth_default

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


### Mount Google Drive

In [2]:
colab_drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATADIR='/content/drive/My Drive/UNM/DrugCentral/data/'
print(f'Files in {DATADIR}')
for dirname, _, filenames in os.walk(f'{DATADIR}'):
  for filename in filenames:
    print(os.path.join(dirname, filename))

Files in /content/drive/My Drive/UNM/DrugCentral/data/
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_structures.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc2023_targets.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_targets_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_targets_unmapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_targets_mapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_drugs_unmapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_drugs_mapped.tsv
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_drugs_mapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_drugs_unmapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight Metadata.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/dc_inx_targets_unmapped.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/Inxight_Db.gsheet
/content/drive/My Drive/UNM/DrugCentral/data/inxight_activity_targets_drugcentral_candidate.gshe

### Authenticate with Google Colab for Google Sheets API access

In [4]:
colab_auth.authenticate_user()
creds, _ = g_auth_default()
gc = gspread.authorize(creds)

### Read data from Google Sheets

*  Inxight Bioactivity
*  DrugCentral Drugs
*  DrugCentral Targets

In [5]:
inx_sheet_url = 'https://docs.google.com/spreadsheets/d/1AK-R0ZqsJwDqOJHtsX0Fud810LQpZMl61tdWAuQNMec/edit'
inx_act = None;
try:
    inx_ss = gc.open_by_url(inx_sheet_url) # Open spreadsheet by URL
    inx_ws_act = inx_ss.worksheet("inxight_activity_targets_drugcentral_candidate") # Select worksheets (specify or use get_worksheet(index))
    inx_act = inx_ws_act.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {inx_sheet_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [6]:
df_inx_act = pd.DataFrame(inx_act[1:], columns=inx_act[0])
df_inx_act.drop(columns=['drug_url', 'evidence_url'], inplace=True)
for tag in df_inx_act.columns:
  df_inx_act.rename(columns={tag:re.sub(r'^', 'inx_', tag)}, inplace=True)
#df_inx_act['inx_target_uniprot'] = df_inx_act['inx_target_uniprot'].replace(r'^\s*$', np.nan)
print(f"Inxight activities: {df_inx_act.shape[0]}; UNIIs: {df_inx_act['inx_unii'].nunique()}; UNIPROTs: {df_inx_act['inx_target_uniprot_id'].nunique()}")
display(df_inx_act.sample(10))

Inxight activities: 18236; UNIIs: 9480; UNIPROTs: 1745


,inx_unii,inx_drug_name,inx_fda_approval_status,inx_fda_approval_year,inx_fda_approved,inx_fda_withdrawn,inx_fda_approval_source_id,inx_fda_approval_source_url,inx_inxight_highest_phase,inx_inxight_development_status_code,...,inx_pharmacology,inx_potency_type,inx_potency_value,inx_potency_unit,inx_target_source,inx_target_chembl_id,inx_chembl_target_type,inx_chembl_lookup_status,inx_target_mapping_status,inx_stitcher_id
6447,CXY7B3Q98Z,ESTRADIOL HEMIHYDRATE,US Previously Marketed,1940,FALSE,FALSE,Dimenformon Dipropionate by Roche-Organon (H.L...,DeHaen 1940-1975 NMEs,Approved,A_0,...,Agonist,EC50,1.4,nM,ChEMBL,CHEMBL242,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,2372.0
10479,LQ7CF8AVM8,MESALAMINE HYDROCHLORIDE,US Approved Rx,1987,TRUE,FALSE,NDA019618,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,EC50,26.0,µM,UniProt,,,,direct_uniprot_single_accession,2980.0
16411,1HJG8C744G,TESOFENSINE TARTRATE MONOHYDRATE,Other,Unknown,FALSE,FALSE,,,Phase II,A_11,...,Blocker,Unknown,,,ChEMBL,CHEMBL238,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,13218.0
18232,R7P8FRP05V,p-Aminophenol,Possibly Marketed Outside US,2013,FALSE,FALSE,BEAUTIFUL WOMANS HAIR LOVES COLORFUL BUBBLES H...,https://dailymed.nlm.nih.gov/dailymed/drugInfo...,,A_7,...,Substrate,Unknown,,,ChEMBL,CHEMBL5101,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,22130.0|16522.0
10814,KE1SEN21RM,MIDOMAFETAMINE,Other,Unknown,FALSE,FALSE,,,,A_11,...,Inhibitor,IC50,1.44,µM,ChEMBL,CHEMBL238,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,22870.0
3412,TY34L4F9OZ,CAPMATINIB,US Approved Rx,2020,TRUE,FALSE,NDA213591,https://www.accessdata.fda.gov/scripts/cder/da...,Phase II,A_0,...,Inhibitor,IC50,0.13,nM,ChEMBL,CHEMBL3717,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,3875.0
6445,CXY7B3Q98Z,ESTRADIOL HEMIHYDRATE,US Previously Marketed,1940,FALSE,FALSE,Dimenformon Dipropionate by Roche-Organon (H.L...,DeHaen 1940-1975 NMEs,Approved,A_0,...,Substrate,Unknown,,,UniProt,,,,direct_uniprot_single_accession,14838.0
7018,GVR41S4ZHJ,FLOSULIDE,Other,Unknown,FALSE,FALSE,,,,A_9,...,Inhibitor,IC50,21.0,nM,ChEMBL,CHEMBL230,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,17923.0
7332,4B93MGE4AL,FUTIBATINIB,US Approved Rx,2022,TRUE,FALSE,NDA214801,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,IC50,1.3,nM,ChEMBL,CHEMBL4142,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,30419.0
16460,5X57I1N37U,TETRABENAZINE METHANESULFONATE,US Approved Rx,2008,TRUE,FALSE,NDA021894,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,Ki,54.0,nM,ChEMBL,CHEMBL1893,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,91.0


In [7]:
display(df_inx_act['inx_target_source'].value_counts(sort=True))

inx_target_source
ChEMBL            14343
UniProt            3885
ChEMBL|UniProt        8
Name: count, dtype: int64

In [8]:
display(df_inx_act['inx_chembl_target_type'].value_counts(sort=True))

inx_chembl_target_type
SINGLE PROTEIN    14351
                   3885
Name: count, dtype: int64

In [9]:
display(df_inx_act['inx_target_organism'].value_counts(sort=True))

inx_target_organism
Homo sapiens                                                                                        14338
Homo sapiens (Human)                                                                                 3256
Rattus norvegicus (Rat)                                                                               161
Mus musculus (Mouse)                                                                                   78
Escherichia coli (strain K12)                                                                          76
                                                                                                    ...  
Hepatitis B virus genotype F2 (isolate Brazil/w4B) (HBV-F)                                              1
Enterobacter cloacae subsp. cloacae (strain ATCC 13047 / DSM 30054 /|||NBRC 13535 / NCDC 279-56)        1
Human immunodeficiency virus type 1 group M subtype B (isolate HXB2)|||(HIV-1)                          1
Vicia sativa subsp. nigra 

In [10]:
display(df_inx_act['inx_pharmacology'].value_counts(sort=True))

inx_pharmacology
Inhibitor                        8230
Agonist                          3686
Antagonist                       3204
Binding Agent                     749
Substrate                         721
Activator                         537
Blocker                           320
Partial Agonist                   261
Modulator                         239
Inverse Agonist                   126
Positive Allosteric Modulator      50
Interacts                          48
Negative Allosteric Modulator      33
Chaperone                          10
Opener                              9
                                    4
Releasing Agent                     4
Neutral Antagonist                  3
Chelating Agent                     2
Name: count, dtype: int64

In [11]:
display(df_inx_act['inx_fda_approval_status'].value_counts(sort=True))

inx_fda_approval_status
Other                             9639
US Previously Marketed            4912
US Approved Rx                    2511
Possibly Marketed Outside US      1154
US Approved Allergenic Extract      20
Name: count, dtype: int64

### Drugs

Read DC Drugs file.

In [12]:
dc_sheet_drug_url = 'https://docs.google.com/spreadsheets/d/1TTUr6L_SVP7w_JQV4HBycR_KAndukrd411zI-Qf-Agw/edit'
dc_drug = None; dc_xref = None;
try:
    dc_drugs_ss = gc.open_by_url(dc_sheet_drug_url) # Open spreadsheet by URL
    dc_ws_drug = dc_drugs_ss.worksheet("dc2023_structures") # Select worksheets (specify or use get_worksheet(index))
    dc_ws_xref = dc_drugs_ss.worksheet("dc2023_xrefs") # Select worksheets (specify or use get_worksheet(index))
    dc_drug = dc_ws_drug.get_all_values() # Get all values (list of lists)
    dc_xref = dc_ws_xref.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {dc_sheet_drug_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [13]:
df_dc_drug = pd.DataFrame(dc_drug[1:], columns=dc_drug[0])
for tag in df_dc_drug.columns:
  df_dc_drug.rename(columns={tag:re.sub(r'^', 'dc_', tag)}, inplace=True)
print(f"DC Structures: {df_dc_drug.shape}")
print(f"DC Structures DC_IDs: {df_dc_drug['dc_id'].nunique()}")
display(df_dc_drug.head())

DC Structures: (4995, 8)
DC Structures DC_IDs: 4995


,dc_id,dc_name,dc_cas_reg_no,dc_smiles,dc_inchikey,dc_inchi,dc_formula,dc_molweight
0,5392,capmatinib,1029712-80-8,CNC(=O)C1=C(C=C(C=C1)C2=NN3C(=CN=C3N=C2)CC4=CC...,LIOLIMKSCNQPLV-UHFFFAOYSA-N,InChI=1S/C23H17FN6O/c1-25-22(31)18-6-5-16(11-1...,C23H17FN6O,412.428
1,5393,selpercatinib,2152628-33-4,CC(C)(COC1=CN2C(=C(C=N2)C#N)C(=C1)C3=CN=C(C=C3...,XIIOFHFUYBLOLW-UHFFFAOYSA-N,"InChI=1S/C29H31N7O3/c1-29(2,37)18-39-24-9-25(2...",C29H31N7O3,525.613
2,5394,ripretinib,1442472-39-0,CCN1C2=CC(=NC=C2C=C(C1=O)C3=CC(=C(C=C3Br)F)NC(...,CEFJVGZHQAGLHS-UHFFFAOYSA-N,InChI=1S/C24H21BrFN5O2/c1-3-31-21-12-22(27-2)2...,C24H21BrFN5O2,510.367
3,5377,molnupiravir,,CC(C)C(=O)OC[C@H]1O[C@H]([C@H](O)[C@@H]1O)N1C=...,HTNPEHXGEKVIHG-QCNRFFRDSA-N,InChI=1S/C13H19N3O7/c1-6(2)12(19)22-5-7-9(17)1...,C13H19N3O7,329.309
4,5395,fluoroestradiol F 18,94153-53-4,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)[1...,KDLLNMRYZGUVMA-ZYMZXAKXSA-N,InChI=1S/C18H23FO2/c1-18-7-6-13-12-5-3-11(20)8...,C18H23FO2,289.381


In [14]:
df_dc_xref = pd.DataFrame(dc_xref[1:], columns=dc_xref[0])
for tag in df_dc_xref.columns:
  df_dc_xref.rename(columns={tag:re.sub(r'^', 'dc_', tag)}, inplace=True)
print(f"DC Xrefs: {df_dc_xref.shape}")
print(f"DC Xrefs DC_IDs: {df_dc_xref['dc_struct_id'].nunique()}")
display(df_dc_xref.head())

DC Xrefs: (82230, 4)
DC Xrefs DC_IDs: 4995


,dc_struct_id,dc_xref_type,dc_xref,dc_dc_struct_name
0,3649,CHEBI,CHEBI:10001,visnadine
1,5100,CHEBI,CHEBI:10014,voacamine
2,1875,CHEBI,CHEBI:100147,nalidixic acid
3,2846,CHEBI,CHEBI:10023,voriconazole
4,659,CHEBI,CHEBI:100241,ciprofloxacin


In [15]:
display(df_dc_xref['dc_xref_type'].value_counts(sort=True))

dc_xref_type
MMSL                           8036
SNOMEDCT_US                    7470
ChEMBL_ID                      7129
NDDF                           5575
UMLSCUI                        5244
UNII                           5185
PUBCHEM_CID                    5013
DRUGBANK_ID                    4368
CHEBI                          4302
INN_ID                         4232
KEGG_DRUG                      4051
VANDF                          3729
RXNORM                         3525
MESH_SUPPLEMENTAL_RECORD_UI    2909
SECONDARY_CAS_RN               2248
IUPHAR_LIGAND_ID               2106
NUI                            2044
MESH_DESCRIPTOR_UI             2014
VUID                           1790
PDB_CHEM_ID                    1260
Name: count, dtype: int64

### DrugCentral struct_id to UNII mappings

Note there are multiple UNIIs for some DC structures, aka chemical entities.

In [16]:
df_dc_unii = df_dc_xref[df_dc_xref['dc_xref_type'] == 'UNII']
df_dc_unii.rename(columns={'dc_xref': 'dc_xref_unii'}, inplace=True)
df_dc_unii.drop(columns=['dc_xref_type'], inplace=True)
df_dc_unii.drop_duplicates(inplace=True, ignore_index=True)
print(f"DrugCentral DC_IDs: {df_dc_unii['dc_struct_id'].nunique()}; UNIIs: {df_dc_unii['dc_xref_unii'].nunique()}")
display(df_dc_unii.head())

DrugCentral DC_IDs: 4931; UNIIs: 5169


,dc_struct_id,dc_xref_unii,dc_dc_struct_name
0,132,001O2254AC,alphaprodine
1,3521,003N66TS6T,rasagiline
2,4769,00435Z54H1,dimethylaminopropionylphenothiazine
3,358,004F72P8F4,bethanechol
4,4988,005990WHZZ,deoxycholic acid


# Merge DC and Inxight - DRUGS - via UNII IDs.

Remove columns related to targets, activities, and FDA.


In [17]:
dc_inx_drug = pd.merge(df_dc_unii, df_inx_act, left_on='dc_xref_unii', right_on='inx_unii', how='left')
for tag in dc_inx_drug.columns:
  if re.search(r'target', tag) or re.search(r'potency', tag) or re.search(r'fda_', tag):
    dc_inx_drug.drop(columns=[tag], inplace=True)
dc_inx_drug.drop(columns=['inx_pharmacology', 'inx_chembl_lookup_status', 'inx_relationship_type',
                          'inx_inxight_development_status_code', 'inx_inxight_highest_phase',
                          'inx_stitcher_id'], inplace=True)
dc_inx_drug.drop_duplicates(inplace=True, ignore_index=True)
print(f"DC-Inxight drug mappings (UNIIs): {dc_inx_drug['dc_xref_unii'].nunique()}")
print(f"DC drugs NOT PRESENT in Inxight: {dc_inx_drug[dc_inx_drug['inx_unii'].isna()]['dc_struct_id'].nunique()}")
print(f"DC drugs PRESENT in Inxight: {dc_inx_drug[dc_inx_drug['inx_unii'].notna()]['dc_struct_id'].nunique()}")
dc_inx_drug = dc_inx_drug[dc_inx_drug['inx_unii'].notna()].reindex() #Remove non-mapped
display(dc_inx_drug.head(10))

DC-Inxight drug mappings (UNIIs): 5169
DC drugs NOT PRESENT in Inxight: 3129
DC drugs PRESENT in Inxight: 1817


,dc_struct_id,dc_xref_unii,dc_dc_struct_name,inx_unii,inx_drug_name
1,3521,003N66TS6T,rasagiline,003N66TS6T,RASAGILINE
3,358,004F72P8F4,bethanechol,004F72P8F4,BETHANECHOL
4,4988,005990WHZZ,deoxycholic acid,005990WHZZ,DEOXYCHOLIC ACID
5,203,00DPD30SOY,amsacrine,00DPD30SOY,AMSACRINE
9,1812,00U7GX0NLM,minaprine,00U7GX0NLM,MINAPRINE
14,1209,01K63SUP8D,fluoxetine,01K63SUP8D,FLUOXETINE
15,2166,01MI4Q9DI3,pilocarpine,01MI4Q9DI3,Pilocarpine
16,4350,01Q9PC255D,ammonium chloride,01Q9PC255D,Ammonium chloride
17,2626,01T23W89FR,thiamylal,01T23W89FR,THIAMYLAL
18,345,01YAE03M7J,betacarotene,01YAE03M7J,BETA CAROTENE


### Produce file for DC with Inxight IDs, URLs, and selected annotations.

In [18]:
dc_inx_drug.to_csv(f"{DATADIR}/dc_inx_drug_mappings.tsv", sep='\t', index=False)

### Targets

Read DC Targets file.

In [19]:
dc_sheet_targets_url = 'https://docs.google.com/spreadsheets/d/1dCnTXIM2C8AOuTVRytWKpJk0FeS9PbW2mneXaWJ6fuc/edit'
dc_targets = None;
try:
    dc_targets_ss = gc.open_by_url(dc_sheet_targets_url) # Open spreadsheet by URL
    dc_ws_targets = dc_targets_ss.worksheet("drugcentral_targets") # Select worksheets (specify or use get_worksheet(index))
    dc_targets = dc_ws_targets.get_all_values() # Get all values (list of lists)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {dc_sheet_targets_url}. Please check the URL and permissions.")
except gspread.exceptions.APIError as e:
    print(f"Error accessing Google Sheets API: {e}. Check permissions and ensure the sheet is shared correctly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [20]:
df_dc_target = pd.DataFrame(dc_targets[1:], columns=dc_targets[0])
print(f"DC Targets: {df_dc_target.shape}")
print(f"DC IDs: {df_dc_target['target_id'].nunique()}; components: {df_dc_target['component_id'].nunique()}")
print(f"DC Target UNIPROTs: {df_dc_target['target_uniprot'].nunique()}")
df_dc_target.drop(columns=['target_id', 'target_name', 'protein_type', 'protein_components', 'swissprot'], inplace=True)
for tag in df_dc_target.columns:
  df_dc_target.rename(columns={tag:re.sub(r'^', 'dc_', tag)}, inplace=True)
display(df_dc_target.sample(10))

DC Targets: (4167, 13)
DC IDs: 3418; components: 3406
DC Target UNIPROTs: 3406


,dc_target_class,dc_component_id,dc_target_uniprot,dc_target_organism,dc_component_name,dc_gene_symbol,dc_geneid,dc_tdl
793,Kinase,2124,Q9HAZ1,Homo sapiens,Dual specificity protein kinase CLK4,CLK4,57396,Tchem
942,Ion channel,1963,Q8NET8,Homo sapiens,Transient receptor potential cation channel su...,TRPV3,162514,Tchem
539,Enzyme,2380,O95263,Homo sapiens,High affinity cAMP-specific and IBMX-insensiti...,PDE8B,8622,Tclin
2989,Ion channel,207,P47870,Homo sapiens,Gamma-aminobutyric acid receptor subunit beta-2,GABRB2,2561,Tclin
4,Enzyme,2201,Q9UGN5,Homo sapiens,Poly [ADP-ribose] polymerase 2,PARP2,10038,Tclin
1181,Transcription factor,1035,P22605,Mus musculus,Retinoic acid receptor beta,Rarb,,
1832,Enzyme,2270,Q9YQ12,Human immunodeficiency virus 1,Protease,protease,,
2913,Enzyme,77,E0VHZ1,Pediculus humanus subsp. corporis,"Cytochrome P450, putative",,,
1955,Ion channel,204,P31644,Homo sapiens,Gamma-aminobutyric acid receptor subunit alpha-5,GABRA5,2558,Tclin
3008,Enzyme,22397,O70282,Rattus norvegicus,Trehalase,Treh,,


# Merge DC and Inxight - TARGETS - via UNIPROT IDs.

Remove columns related to drugs, activities, and FDA.

In [21]:
dc_inx_target = pd.merge(df_dc_target, df_inx_act, left_on='dc_target_uniprot', right_on='inx_target_uniprot_id', how='left')
for tag in dc_inx_target.columns:
  if re.search(r'drug', tag) or re.search(r'potency', tag) or re.search(r'fda_', tag):
    dc_inx_target.drop(columns=[tag], inplace=True)
dc_inx_target.drop(columns=['dc_geneid', 'dc_tdl',
                            'inx_unii', 'inx_pharmacology', 'inx_relationship_type', 'inx_inxight_highest_phase', 'inx_inxight_development_status_code',
                            'inx_chembl_target_type', 'inx_chembl_lookup_status', 'inx_target_mapping_status',
                            'inx_target_gene_id', 'inx_target_label', 'inx_target_gene_symbol', 'inx_target_chembl_id',
                            'inx_target_source', 'inx_target_organism',
                            'inx_stitcher_id'], inplace=True)
dc_inx_target.drop_duplicates(inplace=True, ignore_index=True)
print(f"DC-Inxight target mappings (UNIPROTs): {dc_inx_target['dc_target_uniprot'].nunique()}")
print(f"DC targets NOT PRESENT in Inxight: {dc_inx_target[dc_inx_target['inx_target_uniprot_id'].isna()]['dc_target_uniprot'].nunique()}")
print(f"DC targets PRESENT in Inxight: {dc_inx_target[dc_inx_target['inx_target_uniprot_id'].notna()]['dc_target_uniprot'].nunique()}")
dc_inx_target = dc_inx_target[dc_inx_target['inx_target_uniprot_id'].notna()] #Remove non-mapped
dc_inx_target = dc_inx_target.reindex()
display(dc_inx_target.head(10))

DC-Inxight target mappings (UNIPROTs): 3406
DC targets NOT PRESENT in Inxight: 2296
DC targets PRESENT in Inxight: 1110


,dc_target_class,dc_component_id,dc_target_uniprot,dc_target_organism,dc_component_name,dc_gene_symbol,inx_target_uniprot_id
0,Ion channel,2003,Q92736,Homo sapiens,Ryanodine receptor 2,RYR2,Q92736
4,Enzyme,2201,Q9UGN5,Homo sapiens,Poly [ADP-ribose] polymerase 2,PARP2,Q9UGN5
6,Enzyme,1092,P25092,Homo sapiens,Heat-stable enterotoxin receptor,GUCY2C,P25092
7,GPCR,328,O43613,Homo sapiens,Orexin receptor type 1,HCRTR1,O43613
10,Enzyme,2493,P09467,Homo sapiens,"Fructose-1,6-bisphosphatase 1",FBP1,P09467
12,GPCR,1102,P25116,Homo sapiens,Proteinase-activated receptor 1,F2R,P25116
21,Kinase,1058,P52333,Homo sapiens,Tyrosine-protein kinase JAK3,JAK3,P52333
22,Kinase,1659,Q04912,Homo sapiens,Macrophage-stimulating protein receptor,MST1R,Q04912
23,GPCR,675,P08588,Homo sapiens,Beta-1 adrenergic receptor,ADRB1,P08588
24,GPCR,674,P07550,Homo sapiens,Beta-2 adrenergic receptor,ADRB2,P07550


### Export mapped UniProts

In [22]:
dc_inx_target.to_csv(f"{DATADIR}/dc_inx_target_mapped.tsv", sep='\t', index=False)

# Activities

Produce file of Inxight bioactivities mapped via DC drug and target IDs.
Remove non-essential drug and target variables.

In [23]:
display(df_inx_act.sample(10))

,inx_unii,inx_drug_name,inx_fda_approval_status,inx_fda_approval_year,inx_fda_approved,inx_fda_withdrawn,inx_fda_approval_source_id,inx_fda_approval_source_url,inx_inxight_highest_phase,inx_inxight_development_status_code,...,inx_pharmacology,inx_potency_type,inx_potency_value,inx_potency_unit,inx_target_source,inx_target_chembl_id,inx_chembl_target_type,inx_chembl_lookup_status,inx_target_mapping_status,inx_stitcher_id
3830,5S6VUP419V,CHLORPHENIRAMINE HYDROCHLORIDE,US Previously Marketed,1949,FALSE,FALSE,CHLOR-TRIMETON by SCHERING,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_1,...,Antagonist,IC50,12.0,nM,ChEMBL,CHEMBL231,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,9862.0
12886,G3QE979K1X,PF-03654746,Other,Unknown,FALSE,FALSE,,,Phase II,A_11,...,Antagonist,Unknown,,,ChEMBL,CHEMBL264,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,5306.0
12626,7RN5DR86CK,PAZOPANIB,US Approved Rx,2009,TRUE,FALSE,NDA022465,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,IC50,0.084,µM,ChEMBL,CHEMBL1913,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,5457.0
14966,8054B8370A,SANGUINARIUM CHLORIDE TRIHYDRATE,Other,Unknown,FALSE,FALSE,,,Phase III,A_11,...,Inhibitor,IC50,12.5,µM,ChEMBL,CHEMBL1764941,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,9654.0
5060,554R584P9K,DEXTROMETHORPHAN TANNATE,US Previously Marketed,1954,FALSE,FALSE,Romilar by Hoffmann-La Roche,OB NME Appendix 1950-1985,Approved,A_1,...,Antagonist,Unknown,,,ChEMBL,CHEMBL4787,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,2679.0
8220,LIU00Z1Z84,HYDROCORTISONE HEMISUCCINATE,US Previously Marketed,1951,FALSE,FALSE,HYDROCORTONE by MERCK,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,IC50,0.1,nM,ChEMBL,CHEMBL3070,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,2369.0
9612,SJT761GEGS,LISDEXAMFETAMINE DIMESYLATE,US Previously Marketed,1937,FALSE,FALSE,Dexedrine by Smith Kline French,https://web.archive.org/web/20091219120223/htt...,Approved,A_0,...,Modulator,Unknown,,,ChEMBL,CHEMBL222,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,2590.0
15546,WOX5F63THK,SODIUM TAUROCHOLATE MONOHYDRATE,Other,Unknown,FALSE,FALSE,,,Phase II,A_7,...,Agonist,EC50,4.95,µM,ChEMBL,CHEMBL5409,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,12415.0
13106,96AN057F7A,PHOSPHORYLCHOLINE CHLORIDE,Other,Unknown,FALSE,FALSE,,,,A_11,...,Substrate,Other,,,UniProt,,,,direct_uniprot_single_accession,13198.0
8616,4U4TF1C1F3,INDOMETHACIN MEGLUMINE,US Previously Marketed,1965,FALSE,FALSE,INDOCIN by ZYLA LIFE SCIENCES,https://www.accessdata.fda.gov/scripts/cder/da...,Approved,A_0,...,Inhibitor,IC50,100.0,nM,ChEMBL,CHEMBL1835,SINGLE PROTEIN,ACTIVE,chembl_human_single_protein_to_uniprot,10779.0


In [24]:
inx_act = df_inx_act.copy(deep=True)
tags_act = ['inx_unii', 'inx_drug_name', 'inx_target_uniprot_id', 'inx_target_label', 'inx_pharmacology', ]
for tag in inx_act.columns:
  if tag not in tags_act and not re.search(r'potency', tag):
    inx_act.drop(columns=[tag], inplace=True)
inx_act.drop_duplicates(inplace=True, ignore_index=True)
inx_act = pd.merge(inx_act, df_dc_unii[['dc_struct_id', 'dc_xref_unii']],
                   left_on='inx_unii', right_on='dc_xref_unii', how='inner')
inx_act = pd.merge(inx_act, df_dc_target[['dc_component_id', 'dc_target_uniprot', 'dc_component_name']],
                   left_on='inx_target_uniprot_id', right_on='dc_target_uniprot', how='inner')
inx_act['inx_potency_value'] = pd.to_numeric(inx_act['inx_potency_value'], errors='coerce')
inx_act['inx_potency_unit'] = inx_act['inx_potency_unit'].replace(r'^\s*$', np.nan, regex=True)
print(f"Inx activity data mapped to DC drugs and targets, drugs: {inx_act['dc_struct_id'].nunique()}; targets: {inx_act['dc_target_uniprot'].nunique()}")
print(f"Inx activity data mapped to DC drugs and targets, rows: {inx_act.shape[0]}; potency values: {inx_act['inx_potency_value'].notna().sum()}")
display(inx_act.sample(10))

Inx activity data mapped to DC drugs and targets, drugs: 1701; targets: 666
Inx activity data mapped to DC drugs and targets, rows: 5525; potency values: 3664


,inx_unii,inx_drug_name,inx_target_uniprot_id,inx_target_label,inx_pharmacology,inx_potency_type,inx_potency_value,inx_potency_unit,dc_struct_id,dc_xref_unii,dc_component_id,dc_target_uniprot,dc_component_name
2169,B4SF212641,FOSPHENYTOIN,Q9UQD0,Sodium channel protein type VIII alpha subunit,Inhibitor,EC50,20.00,µM,1247,B4SF212641,2228,Q9UQD0,Sodium channel protein type 8 subunit alpha
712,3G6A5W338E,CAFFEINE,P30542,Adenosine A1 receptor,Antagonist,Ki,44.90,µM,463,3G6A5W338E,1178,P30542,Adenosine receptor A1
4883,M0XW1UBI14,TESTOSTERONE CYPIONATE,P10275,Androgen Receptor,Agonist,EC50,3.16,nM,4454,M0XW1UBI14,815,P10275,Androgen receptor
218,8110R61I4U,AMISULPRIDE,P25100,Alpha-1D adrenergic receptor,Binding Agent,IC50,320.00,nM,179,8110R61I4U,746,P25100,Alpha-1D adrenergic receptor
3725,9647FM7Y3Z,PANOBINOSTAT,Q92769,Histone deacetylase 2,Inhibitor,IC50,13.00,nM,4682,9647FM7Y3Z,282,Q92769,Histone deacetylase 2
4707,V99T50803M,SUNITINIB,P16234,Platelet-derived growth factor receptor alpha,Inhibitor,Kd,0.79,nM,2544,V99T50803M,760,P16234,Platelet-derived growth factor receptor alpha
886,8R1V1STN48,CIANIDANOL,P35354,Cyclooxygenase-2,Inhibitor,IC50,93.30,µM,629,8R1V1STN48,2617,P35354,Prostaglandin G/H synthase 2
1299,NQO8R319LY,DIPHENIDOL,P08172,Muscarinic acetylcholine receptor M2,Antagonist,pKi,5.55,NaN,313,NQO8R319LY,693,P08172,Muscarinic acetylcholine receptor M2
749,97J7NP0XJY,CARAMIPHEN,P11229,Muscarinic acetylcholine receptor M1,Antagonist,Ki,1.20,nM,486,97J7NP0XJY,696,P11229,Muscarinic acetylcholine receptor M1
2946,R4K19W6S7Q,MABUTEROL,P07550,Beta-2 adrenergic receptor,Agonist,Unknown,NaN,NaN,1623,R4K19W6S7Q,674,P07550,Beta-2 adrenergic receptor


In [25]:
inx_act['inx_pharmacology'].value_counts()

inx_pharmacology
Inhibitor                        2144
Agonist                          1442
Antagonist                       1233
Substrate                         159
Blocker                           133
Binding Agent                     102
Partial Agonist                    85
Activator                          80
Modulator                          72
Inverse Agonist                    35
Positive Allosteric Modulator      13
Negative Allosteric Modulator      13
Interacts                           6
Opener                              5
Chaperone                           2
                                    1
Name: count, dtype: int64

In [26]:
inx_act.to_csv(f"{DATADIR}/inx_act_mapped.tsv", sep="\t", index=False)